# P model: TensorFlow CPU vs GPU vs PyTorch

This notebook **does not** hide TensorFlow GPUs (`set_visible_devices`). It runs the **same** Keras `modelP` forward pass on:

- **TF /CPU:0** — reference device for parity
- **TF /GPU:0** — if a GPU is visible to TensorFlow (may match CPU or diverge depending on stack/driver)
- **PyTorch** — on **CUDA** if available, else CPU

SeisBench windows match `tf_pt_working/tf_pt_p_waveform_picking_demo.ipynb` (P-centered 6000 samples). Each row plots **TF CPU**, **TF GPU** (if run), and **PT** P-probabilities.

If `max|TF_cpu − TF_gpu|` is large, the issue is TensorFlow device math (known to bite this graph on some setups), not PT weights.

### If `TensorFlow sees GPU devices: []` but you have NVIDIA GPUs

That almost always means the **TensorFlow pip package** in this environment is **CPU-only**, or its bundled CUDA does not match your **driver** — not that the hardware is absent. **PyTorch** bundles its own CUDA and can work while **TF** does not. Fix by installing a **GPU-enabled** TensorFlow build per [TensorFlow install](https://www.tensorflow.org/install/pip) (e.g. `pip install -U 'tensorflow[and-cuda]'` for supported versions). The model cell imports **TensorFlow before PyTorch** to reduce duplicate cuDNN registration warnings.

`pip install 'setuptools>=69,<81'` if SeisBench fails on `pkg_resources`.


### Two GPUs: do TensorFlow and PyTorch need different cards?

**Usually no.** If `TensorFlow sees GPU devices: []`, TensorFlow is **not** failing because PyTorch “took” both GPUs. In that state TF never attached to the driver for GPU compute; fixing it is about **TensorFlow + driver/CUDA** (see [install](https://www.tensorflow.org/install/pip)), not about assigning one framework per GPU.

**After** TensorFlow lists GPUs, you *can* avoid sharing one GPU between TF and PT (memory, contention) by giving **this notebook process** only one card, **before** any `import tensorflow` / `import torch` — set `CUDA_VISIBLE_DEVICES` and **restart the kernel**:

```python
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # or "1" — only that physical GPU is visible
```

Then TF and PT both see a single logical `GPU:0` (the physical index you chose). Using **`"0,1"`** keeps both visible; use device placement (`tf.device`, `torch.cuda.set_device`) if you need to split — but that does **not** fix an empty TF GPU list.

**Ada (RTX 6000 Ada):** ensure your **NVIDIA driver** is new enough for the CUDA version TensorFlow was built with; a mismatched driver often yields **no** TF GPUs while PyTorch still works (different CUDA runtime bundled with torch).


### TensorFlow shows `GPU devices: []` but the terminal works

This is almost always the **Jupyter kernel** using a **different Python** than `conda activate eqcctpro` in the shell. Check the printed **`Kernel interpreter:`** path — it should match `which python` in the env where GPU works.

From the repo root, run:

```bash
PYTHONPATH=. python -m validation.tf_gpu_env_sniff
```

If that lists GPUs, select the notebook kernel whose interpreter is the **same** `sys.executable` (Cursor → kernel picker → *Python (eqcctpro)* or *Browse* → `.../envs/eqcctpro/bin/python`). You can register the kernel once with:

```bash
conda activate eqcctpro
python -m ipykernel install --user --name eqcctpro --display-name "Python (eqcctpro)"
```

No package downgrade is required — this is environment selection, not version mismatch.


In [ ]:
from pathlib import Path
import os
import sys

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
elif REPO.name == "tf_pt_working":
    REPO = REPO.parent.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

P_H5 = REPO / "ModelPS" / "test_trainer_024.h5"
S_H5 = REPO / "ModelPS" / "test_trainer_021.h5"
PT_CKPT = REPO / "ModelPS" / "eqcct_model_p.pt"

print("REPO", REPO)
print("Kernel interpreter:", sys.executable)
print("(If TensorFlow shows no GPUs here but works in a terminal, the Jupyter kernel is usually a *different* Python — select the same env, e.g. eqcctpro, in the kernel picker.)")
print("P_H5 exists", P_H5.is_file(), "S_H5 exists", S_H5.is_file())
print("PT_CKPT exists", PT_CKPT.is_file())

import numpy as np

# Log level before importing TensorFlow
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# Import TensorFlow before matplotlib/torch — then list GPUs immediately.
import tensorflow as tf

gpus_tf = tf.config.list_physical_devices("GPU")
print("TensorFlow sees GPU devices (before matplotlib/torch):", gpus_tf)

import matplotlib.pyplot as plt
import torch

print("TensorFlow version:", tf.__version__)
try:
    bi = tf.sysconfig.get_build_info()
    print("TF build cuda_version:", bi.get("cuda_version"))
    print("TF build cudnn_version:", bi.get("cudnn_version"))
except Exception as e:
    print("TF build info:", e)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
if torch.cuda.is_available():
    print("PyTorch CUDA:", torch.cuda.device_count(), "device(s),", torch.cuda.get_device_name(0))
else:
    print("PyTorch CUDA: not available")

if not gpus_tf:
    print(
        "[hint] Same TF build can list GPUs in a terminal but [] in Jupyter if the kernel "
        "uses another Python. Compare `sys.executable` above to: "
        "`python -m validation.tf_gpu_env_sniff` run from the repo with PYTHONPATH set. "
        "Register kernel:  python -m ipykernel install --user --name eqcctpro "
        '--display-name "Python (eqcctpro)"'
    )

for g in gpus_tf:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

from reference.predictor_tf import load_eqcct_model
from models.predictor_pt_p import EQCCTModelP
from conversion.loader import load_eqcct_model_p_weights

model_p_tf, _ = load_eqcct_model(str(P_H5), str(S_H5))

model_pt = EQCCTModelP()
model_pt.eval()
if PT_CKPT.is_file():
    try:
        ckpt = torch.load(PT_CKPT, map_location="cpu", weights_only=False)
    except TypeError:
        ckpt = torch.load(PT_CKPT, map_location="cpu")
    model_pt.load_state_dict(ckpt["state_dict"])
    print("Loaded PyTorch from", PT_CKPT)
else:
    load_eqcct_model_p_weights(model_pt, h5_path=str(P_H5))
    print("Loaded PyTorch weights from H5", P_H5)

_pt_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_pt.to(_pt_dev)
print("PyTorch device:", _pt_dev)
print("TF / PT P models ready.")


In [ ]:
import random

import seisbench.data as sbd

SAMPLE_RATE = 100
DATASET_CHOICE = 1  # 0 = TXED, 1 = STEAD
N_SHUFFLE_SEEDS = 10
SHUFFLE_SEEDS = list(range(N_SHUFFLE_SEEDS))

if DATASET_CHOICE == 1:
    dataset = sbd.STEAD(sampling_rate=SAMPLE_RATE, component_order="ZNE")
    ds_name = "STEAD"
else:
    dataset = sbd.TXED(sampling_rate=SAMPLE_RATE, component_order="ZNE")
    ds_name = "TXED"

md = dataset.metadata
good_p = md["trace_p_arrival_sample"].notna() & (md["trace_p_arrival_sample"] > 0)
valid_names = md[good_p]["trace_name"].tolist()
print(f"{ds_name}: {len(valid_names)} traces with P pick")


def norm_std_time(x: np.ndarray) -> np.ndarray:
    m = x.mean(axis=1, keepdims=True)
    s = x.std(axis=1, keepdims=True) + 1e-8
    return ((x - m) / s).astype(np.float32)


def window_for_shuffle_seed(shuffle_seed: int):
    names = list(valid_names)
    random.seed(shuffle_seed)
    random.shuffle(names)
    for trace_name in names:
        idx = dataset.get_idx_from_trace_name(trace_name)
        raw = np.asarray(dataset.get_waveforms([idx])[0], dtype=np.float32)
        if raw.shape[1] < 6000:
            continue
        p_sample = int(md.loc[idx, "trace_p_arrival_sample"])
        start = max(0, min(p_sample - 3000, raw.shape[1] - 6000))
        win = raw[:, start : start + 6000]
        wf = norm_std_time(win.transpose(1, 0)[np.newaxis, ...])
        p_in_window = p_sample - start
        return trace_name, wf, p_in_window
    raise RuntimeError("No trace ≥ 6000 samples in shuffled list")


In [ ]:
def _tf_to_numpy(out):
    if hasattr(out, "numpy"):
        arr = out.numpy()
    else:
        arr = np.asarray(out)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr


def tf_p_on_device(wf_np, device: str):
    x = tf.constant(wf_np, dtype=tf.float32)
    with tf.device(device):
        out = model_p_tf(x, training=False)
    return _tf_to_numpy(out)


HAS_TF_GPU = len(gpus_tf) > 0

rows = []
for shuffle_seed in SHUFFLE_SEEDS:
    trace_name, wf, p_in_window = window_for_shuffle_seed(shuffle_seed)
    p_cpu = tf_p_on_device(wf, "/CPU:0")
    p_gpu = tf_p_on_device(wf, "/GPU:0") if HAS_TF_GPU else None
    with torch.no_grad():
        t = torch.from_numpy(wf).to(_pt_dev)
        p_pt = model_pt(t).cpu().numpy()
        if p_pt.ndim == 3:
            p_pt = p_pt[..., 0]
    d_cpu_gpu = (
        float(np.abs(p_cpu - p_gpu).max()) if p_gpu is not None else float("nan")
    )
    d_cpu_pt = float(np.abs(p_cpu - p_pt).max())
    d_gpu_pt = (
        float(np.abs(p_gpu - p_pt).max()) if p_gpu is not None else float("nan")
    )
    rows.append(
        {
            "seed": shuffle_seed,
            "trace": trace_name,
            "p_cat": p_in_window,
            "p_tf_cpu": p_cpu,
            "p_tf_gpu": p_gpu,
            "p_pt": p_pt,
            "max_cpu_gpu": d_cpu_gpu,
            "max_cpu_pt": d_cpu_pt,
            "max_gpu_pt": d_gpu_pt,
            "argmax_cpu": int(p_cpu[0].argmax()),
            "argmax_gpu": int(p_gpu[0].argmax()) if p_gpu is not None else -1,
            "argmax_pt": int(p_pt[0].argmax()),
        }
    )

for r in rows:
    gpu_note = (
        f"  max|cpu-gpu|={r['max_cpu_gpu']:.4f}"
        if not np.isnan(r["max_cpu_gpu"])
        else ""
    )
    print(
        f"seed {r['seed']:2d}{gpu_note}  max|cpu-PT|={r['max_cpu_pt']:.4f}"
        + (
            f"  max|gpu-PT|={r['max_gpu_pt']:.4f}"
            if not np.isnan(r["max_gpu_pt"])
            else ""
        )
        + f"  argmax cpu={r['argmax_cpu']:4d} gpu={r['argmax_gpu']:4d} PT={r['argmax_pt']:4d}  cat_P={r['p_cat']:4d}"
    )

fig_h = max(2.2, 2.1 * N_SHUFFLE_SEEDS)
fig, axes = plt.subplots(N_SHUFFLE_SEEDS, 1, figsize=(12, fig_h), sharex=True, layout="constrained")
if N_SHUFFLE_SEEDS == 1:
    axes = [axes]
for ax, r in zip(axes, rows):
    ax.plot(r["p_tf_cpu"][0], label="TF P (CPU)", alpha=0.9)
    if r["p_tf_gpu"] is not None:
        ax.plot(r["p_tf_gpu"][0], label="TF P (GPU)", alpha=0.85)
    ax.plot(r["p_pt"][0], label=f"PT P ({_pt_dev.type})", alpha=0.75)
    ax.axvline(r["p_cat"], color="gray", ls=":", alpha=0.65)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel(f"seed {r['seed']}", fontsize=9)
    ax.tick_params(labelsize=8)
axes[0].legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("sample")
title = f"{ds_name}: TF CPU vs TF GPU vs PT — P probabilities ({N_SHUFFLE_SEEDS} seeds)"
if not HAS_TF_GPU:
    title += " [no TF GPU — CPU vs PT only]"
fig.suptitle(title, fontsize=11)
plt.show()
